# DengueRadar — 02 | Modelagem, comparação e avaliação crítica

**FIAP · Data Science · CitizenTech**

## 1. Problema e objetivo
A dengue é um desafio sazonal da saúde pública: variações na quantidade de casos podem afetar cidadãos, equipes assistenciais e gestores responsáveis pelo planejamento de ações. Neste trabalho, investigamos se dados históricos de notificações e informações meteorológicas ajudam a estimar as contagens semanais de casos.

O objetivo não é afirmar que prevemos surtos confirmados ou focos individuais. Com as entradas climáticas da própria semana, o modelo representa uma **estimativa condicional / análise retrospectiva e de cenários**, e só poderia ser usado para previsão futura após definir entradas efetivamente disponíveis no instante da previsão (por exemplo, previsões meteorológicas) e validar essa aplicação.

## 2. Fontes, alvo e features
O dataset tratado foi produzido no notebook 01, a partir de **SINAN/DATASUS (via TABNET)** e **INMET**, com alinhamento por ano e semana.

| Coluna | Papel | Interpretação |
|---|---|---|
| `casos` | **Variável-alvo (`y`)** | Casos notificados na semana segundo o recorte da exportação |
| `semana_epidem` | Feature | Semana do ano; informação sazonal |
| `temp_media` | Feature | Temperatura média semanal, em °C |
| `precipitacao_total` | Feature | Precipitação total semanal, em mm |
| `ano` | Controle cronológico | Ordenação e documentação; não é feature |

**Nota sobre vazamento:** os dados são ordenados temporalmente, nenhuma linha de teste participa do ajuste dos modelos/seleção de hiperparâmetros e a comparação final usa o teste somente uma vez. Não fazemos preenchimento usando estatísticas futuras. Como o clima é contemporâneo ao alvo, a análise não deve ser chamada de previsão operacional de uma semana futura sem uma fonte de clima futuro.

In [4]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path.cwd().parent if Path.cwd().name.lower() in ('notebooks', 'notebook') else Path.cwd()
DATA = BASE / 'data' / 'processed' / 'dataset_dengueradar_limpo.csv'
MODELS = BASE / 'models'
MODELS.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {DATA}. Execute antes o notebook 01 com os CSVs reais.')
df = pd.read_csv(DATA)
FEATURES = ['semana_epidem', 'temp_media', 'precipitacao_total']
REQUIRED = ['ano', 'semana_epidem', 'temp_media', 'precipitacao_total', 'casos']
missing = set(REQUIRED) - set(df.columns)
if missing:
    raise ValueError(f'Colunas ausentes: {sorted(missing)}')
for coluna in REQUIRED:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')
if df[REQUIRED].isna().any().any():
    raise ValueError('Existem nulos ou valores inválidos. Revise a limpeza no notebook 01.')
if df.duplicated(['ano','semana_epidem']).any():
    raise ValueError('Períodos ano/semana duplicados.')
if ((df['casos'] < 0) | (df['precipitacao_total'] < 0)).any():
    raise ValueError('Casos e precipitação devem ser não negativos.')
if not df['semana_epidem'].between(1,53).all():
    raise ValueError('Semana epidemiológica fora do intervalo 1..53.')
df = df.sort_values(['ano','semana_epidem']).reset_index(drop=True)
if len(df) < 45:
    raise ValueError('São necessárias pelo menos 45 observações para esta divisão exploratória em três blocos.')
print(f'Dataset: {len(df)} semanas | período {int(df.ano.min())}–{int(df.ano.max())}')
print('Features:', FEATURES, '| Alvo: casos')
display(df.head())

ModuleNotFoundError: No module named 'matplotlib'

## 3. Divisão cronológica em treino, validação e teste
Usamos blocos consecutivos: **60% para treino, 20% para validação e os 20% mais recentes para teste**. A validação serve para comparar os dois modelos; o teste é reservado para a avaliação final, sem participação na escolha. O cálculo dos tamanhos preserva pelo menos oito semanas em cada bloco final.

*Observação:* uma única divisão temporal é mais adequada que embaralhamento aleatório, mas ainda não substitui um backtesting com múltiplas origens e anos epidemiológicos.

In [ ]:
n_total = len(df)
n_test = max(8, int(np.ceil(n_total * 0.20)))
n_val = max(8, int(np.ceil(n_total * 0.20)))
n_train = n_total - n_val - n_test
if n_train < 20:
    raise ValueError('Treino muito curto após a divisão. Amplie o histórico.')
train = df.iloc[:n_train].copy()
val = df.iloc[n_train:n_train+n_val].copy()
test = df.iloc[n_train+n_val:].copy()
assert len(train) + len(val) + len(test) == n_total
assert train.index.max() < val.index.min() < test.index.min()

def periodo(frame):
    return f"{int(frame.iloc[0]['ano'])}/SE{int(frame.iloc[0]['semana_epidem']):02d} a {int(frame.iloc[-1]['ano'])}/SE{int(frame.iloc[-1]['semana_epidem']):02d}"
print(f'Treino: {len(train)} semanas ({periodo(train)})')
print(f'Validação: {len(val)} semanas ({periodo(val)})')
print(f'Teste: {len(test)} semanas ({periodo(test)})')

fig, ax = plt.subplots(figsize=(11, 1.8))
ax.barh(['Períodos'], [len(train)], label='Treino')
ax.barh(['Períodos'], [len(val)], left=[len(train)], label='Validação')
ax.barh(['Períodos'], [len(test)], left=[len(train)+len(val)], label='Teste')
ax.set_xlabel('Observações em ordem cronológica')
ax.legend(loc='upper center', bbox_to_anchor=(.5, 1.55), ncol=3)
plt.tight_layout(); plt.show()

NameError: name 'df' is not defined

## 4. Dois modelos treinados e baseline de referência
Treinamos **Random Forest Regressor** (modelo principal) e **Decision Tree Regressor** (modelo comparativo simples) usando somente o bloco de treino. Acrescentamos uma **referência sazonal** baseada na mediana histórica da semana, sem ajuste de um segundo estimador, para verificar se a complexidade do Machine Learning realmente traz ganho.

Os hiperparâmetros são predefinidos neste estudo exploratório; não utilizamos resultados do teste para alterá-los.

In [ ]:
modelos = {
    'Random Forest': RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1),
    'Árvore de Decisão': DecisionTreeRegressor(max_depth=6, min_samples_leaf=2, random_state=42),
}
for nome, modelo in modelos.items():
    modelo.fit(train[FEATURES], train['casos'])
    print(f'{nome}: treinamento concluído usando {len(train)} observações.')

def baseline_sazonal(train_ref, frame):
    semanas = train_ref.groupby('semana_epidem')['casos'].median()
    mediana_global = float(train_ref['casos'].median())
    return frame['semana_epidem'].map(semanas).fillna(mediana_global).to_numpy(dtype=float)

def metricas(y_real, y_prev):
    y_real = np.asarray(y_real, dtype=float)
    y_prev = np.asarray(y_prev, dtype=float)
    mae = float(mean_absolute_error(y_real,y_prev))
    rmse = float(np.sqrt(mean_squared_error(y_real,y_prev)))
    r2 = float(r2_score(y_real,y_prev)) if len(y_real) >= 2 else float('nan')
    total = float(np.sum(np.abs(y_real)))
    wape = 100 * float(np.sum(np.abs(y_real-y_prev))) / total if total else float('nan')
    return {'MAE':mae,'RMSE':rmse,'R²':r2,'WAPE (%)':wape}

pred_val = {nome:np.maximum(0,modelo.predict(val[FEATURES])) for nome,modelo in modelos.items()}
pred_val['Baseline sazonal'] = baseline_sazonal(train,val)
metricas_val = pd.DataFrame({nome:metricas(val['casos'],prev) for nome,prev in pred_val.items()}).T
metricas_val = metricas_val.sort_values('MAE')
print('Desempenho na VALIDAÇÃO (não é o teste):')
display(metricas_val.round(3))
melhor_validacao = min(modelos.keys(), key=lambda nome:metricas_val.loc[nome,'MAE'])
print('Melhor modelo treinado na validação por MAE:', melhor_validacao)
print('Nota: a comparação com baseline também deve ser observada; baixo erro não comprova antecipação de surtos.')

## 5. Avaliação final dos modelos (teste nunca visto)
Depois da seleção pela validação, treinamos novamente os dois modelos **somente com treino + validação** e avaliamos cada um no bloco de teste. A referência sazonal é calculada usando somente esse histórico anterior. Exibimos **MAE, RMSE e R²** (além de WAPE auxiliar).

- **MAE:** erro médio em quantidade de casos (menor é melhor).
- **RMSE:** penaliza erros grandes, relevantes nos picos de casos (menor é melhor).
- **R²:** compara os erros à variação dos dados de teste; pode ser negativo, sobretudo em períodos curtos ou com baixa variabilidade (maior é melhor).

In [ ]:
train_val = pd.concat([train, val], ignore_index=True)
modelos_teste = {
    'Random Forest': RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1),
    'Árvore de Decisão': DecisionTreeRegressor(max_depth=6, min_samples_leaf=2, random_state=42),
}
for nome, modelo in modelos_teste.items():
    modelo.fit(train_val[FEATURES], train_val['casos'])
pred_test = {nome:np.maximum(0,modelo.predict(test[FEATURES])) for nome,modelo in modelos_teste.items()}
pred_test['Baseline sazonal'] = baseline_sazonal(train_val,test)
metricas_teste = pd.DataFrame({nome:metricas(test['casos'],prev) for nome,prev in pred_test.items()}).T
print('Métricas no TESTE cronológico:')
display(metricas_teste.round(3))

avaliacao = test[['ano','semana_epidem','casos']].copy()
avaliacao['previsto_rf'] = pred_test['Random Forest']
avaliacao['previsto_arvore'] = pred_test['Árvore de Decisão']
avaliacao['previsto_baseline'] = pred_test['Baseline sazonal']
avaliacao.to_csv(MODELS / 'avaliacao_temporal.csv', index=False)
print('Avaliação salva em:', MODELS / 'avaliacao_temporal.csv')

## 6. Visualizações explicativas
Apresentamos: (a) importância global das features no Random Forest; (b) resíduos no teste, definidos como **casos observados menos estimados**; (c) curva cronológica de casos observados e previstos; e (d) comparação direta das métricas.

**Interpretação:** a importância por impureza indica como o algoritmo utilizou as variáveis e não determina causalidade. Um erro elevado em picos epidemiológicos pode ser particularmente relevante mesmo com MAE moderado.

In [ ]:
importancias = pd.Series(modelos_teste['Random Forest'].feature_importances_, index=FEATURES).sort_values()
residuos_rf = test['casos'].to_numpy() - pred_test['Random Forest']

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(importancias.index, importancias.values)
ax.set_title('Random Forest — importância das variáveis (por impureza)')
ax.set_xlabel('Importância relativa')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(pred_test['Random Forest'], residuos_rf, alpha=0.75)
ax.axhline(0, linestyle='--', linewidth=1.3)
ax.set(title='Resíduos no teste cronológico', xlabel='Casos estimados pelo Random Forest', ylabel='Observado − Estimado (casos)')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(test))
ax.plot(x, test['casos'].to_numpy(), marker='o', label='Observado')
ax.plot(x, pred_test['Random Forest'], marker='.', label='Random Forest')
ax.plot(x, pred_test['Árvore de Decisão'], marker='.', label='Árvore de Decisão')
ax.set(title='Casos observados versus estimados — semanas de teste', xlabel='Observações no teste (ordem cronológica)', ylabel='Casos')
ax.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.3))
metricas_teste[['MAE','RMSE']].plot(kind='bar', ax=ax)
ax.set(title='Comparação de erro no teste', ylabel='Casos', xlabel='Modelo')
ax.tick_params(axis='x',rotation=0)
plt.tight_layout(); plt.show()

## 7. Conclusão técnica — comparação fundamentada nos resultados
A conclusão abaixo é gerada **a partir das métricas calculadas**, sem antecipar um vencedor. A preferência na validação segue MAE, enquanto o desempenho no teste é relatado como evidência independente. Se o baseline sazonal superar os modelos, isso é uma limitação importante, não um resultado a esconder.

In [ ]:
melhor_teste = metricas_teste['MAE'].idxmin()
mae_rf = float(metricas_teste.loc['Random Forest','MAE'])
mae_arvore = float(metricas_teste.loc['Árvore de Decisão','MAE'])
mae_baseline = float(metricas_teste.loc['Baseline sazonal','MAE'])
print('CONCLUSÃO COM BASE NOS DADOS DESTE TESTE')
print(f'Na validação, o modelo supervisionado com menor MAE foi: {melhor_validacao}.')
print(f'No teste cronológico, o menor MAE foi obtido por: {melhor_teste}.')
print(f'MAE no teste (casos): RF={mae_rf:.2f} | Árvore={mae_arvore:.2f} | Baseline={mae_baseline:.2f}.')
print('A interpretação deve considerar também RMSE e R², a quantidade de semanas e os erros nos picos.')
if melhor_teste == 'Random Forest':
    print('Nesta janela, o Random Forest apresentou o menor MAE, mas isso não garante superioridade em outros anos.')
elif melhor_teste == 'Árvore de Decisão':
    print('Nesta janela, a Árvore de Decisão superou o Random Forest em MAE; a escolha do RF para implantação não deve ser justificada como superior por esta avaliação.')
else:
    print('O baseline sazonal foi melhor em MAE: os modelos mais complexos não justificaram ganho nesta janela de teste.')
print('Limite: uma avaliação retrospectiva com clima do mesmo período não valida previsões futuras de surtos.')

## 8. Texto analítico e reflexão crítica
**Nossas escolhas técnicas:** adotamos Random Forest por sua capacidade de representar relações não lineares e interações entre variáveis, e uma árvore de decisão como segunda referência treinada. O baseline sazonal verifica se o aprendizado de sazonalidade isoladamente já explica parte importante do resultado. Separar treino, validação e teste cronologicamente reduz a contaminação causada por amostragem aleatória de semanas.

**Limitações e possíveis vieses:**
1. Notificações do SINAN/DATASUS podem sofrer **subnotificação, atrasos e revisão retrospectiva**; números recentes não necessariamente são consolidados.
2. A área e o período da extração devem ser consistentes com os dados climáticos do INMET. Uma estação isolada pode não representar toda a área estudada.
3. A associação entre clima e casos pode apresentar **defasagens** (por exemplo, ciclo do vetor, incubação e atraso de registro), não modeladas nas features atuais.
4. A semana do ano pode dominar a previsão; importância elevada não significa relação causal e pode esconder dificuldade de generalização entre temporadas.
5. Uma divisão temporal única pode ser sensível ao ano e à intensidade da epidemia. Idealmente, ampliar o histórico e fazer *backtesting* com várias janelas.
6. As variáveis climáticas da mesma semana podem não estar disponíveis para uma previsão antecipada; o simulador oferece **cenários hipotéticos condicionais**, não prova de impacto causal.
7. Datas ausentes removidas na união podem produzir seleção de semanas e viés de cobertura.

**Próximas evoluções:** testar defasagens de casos e clima, incorporar denominadores populacionais válidos, avaliar outras estações/regiões, documentar o horizonte de previsão e verificar desempenho especialmente em picos. A solução é um **protótipo acadêmico de apoio analítico**, não substitui recomendações oficiais de vigilância.

## 9. Exportação dos artefatos para o Streamlit
Mantemos o arquivo `models/modelo_rf_dengue.pkl` e a lista `features_dengue.pkl` para compatibilidade com o aplicativo existente. O modelo de implantação é reajustado com todo o histórico **somente depois de concluir a avaliação temporal**. Esse reajuste **não** muda as métricas calculadas no teste. Também exportamos o modelo comparativo e os metadados de avaliação.

**Nota:** se a Árvore de Decisão superar o Random Forest, o notebook registra isso de forma transparente; o aplicativo continua carregando o Random Forest por compatibilidade, sem afirmar que foi o melhor.

In [ ]:
modelo_final_rf = RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1)
modelo_final_rf.fit(df[FEATURES],df['casos'])
modelo_final_arvore = DecisionTreeRegressor(max_depth=6, min_samples_leaf=2, random_state=42)
modelo_final_arvore.fit(df[FEATURES],df['casos'])
joblib.dump(modelo_final_rf, MODELS/'modelo_rf_dengue.pkl')
joblib.dump(modelo_final_arvore, MODELS/'modelo_arvore_dengue.pkl')
joblib.dump(FEATURES, MODELS/'features_dengue.pkl')

# JSON serializável com os resultados efetivamente produzidos no notebook.
def dict_metricas(tabela):
    return {str(nome):{str(k):float(v) if np.isfinite(v) else None for k,v in valores.items()}
            for nome, valores in tabela.iterrows()}

metadata = {
 'modelo': 'RandomForestRegressor',
 'modelo_comparativo': 'DecisionTreeRegressor',
 'validacao': 'treino 60%, validacao 20%, teste 20% em blocos cronologicos',
 'features': FEATURES, 'alvo':'casos',
 'registros_totais':int(len(df)), 'registros_treino':int(len(train)),
 'registros_validacao':int(len(val)), 'registros_teste':int(len(test)),
 'treino_periodo':periodo(train), 'validacao_periodo':periodo(val),'teste_periodo':periodo(test),
 'teste_inicio':{'ano':int(test.iloc[0].ano),'semana':int(test.iloc[0].semana_epidem)},
 'teste_fim':{'ano':int(test.iloc[-1].ano),'semana':int(test.iloc[-1].semana_epidem)},
 'metricas_validacao':dict_metricas(metricas_val),
 'metricas_teste':dict_metricas(metricas_teste),
 'metricas_rf_teste': {k:round(float(v),3) if np.isfinite(v) else None for k,v in metricas_teste.loc['Random Forest'].items()},
 'metricas_baseline_teste': {k:round(float(v),3) if np.isfinite(v) else None for k,v in metricas_teste.loc['Baseline sazonal'].items()},
 'modelo_escolhido_na_validacao_por_mae':melhor_validacao,
 'menor_mae_no_teste':melhor_teste,
 'observacao':'Métricas do bloco de teste cronológico antes do ajuste final. Clima da própria semana: estudo retrospectivo / cenário condicional, não previsão operacional comprovada.'
}
(MODELS/'avaliacao_modelo.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Artefatos exportados:', MODELS)
print('- modelo_rf_dengue.pkl, modelo_arvore_dengue.pkl, features_dengue.pkl')
print('- avaliacao_temporal.csv, avaliacao_modelo.json')